In [1]:
import math
import random
import numpy as np
import matplotlib.pyplot as plt # THIS ISN't FUNCTIONAL, IT's JUST TEMPORARY
import time
from waveNetArchitecture import Value, Linear, BatchNorm1D, Tanh, Embedding, FlattenConsecutive, Sequential
words = open('names.txt', 'r').read().splitlines()

In [2]:
stoi = {s:i for i,s in enumerate(sorted(list(set(''.join(words)))))}
stoi['.']=26
itos = {i:s for s,i in stoi.items()}

blockSize = 8 # Increase the amount of context
inputs, outputs = [],[]

In [3]:
for w in words:
    context = [0] * blockSize # start with 3 starting characters
    for ch in w + '.':
        ix = stoi[ch] # get the current character
        inputs.append(context) # add input to inputs, output to outputs
        outputs.append(ix)
        context = context[1:] + [ix] # update the context

In [4]:
n1 = int(.8 * len(inputs))
n2 = int(.9 * len(inputs))

trainIn = inputs[:n1] # Training data (80%)
trainOut = outputs[:n1]
devIn = inputs[n1:n2] # Development data (10%) For use on like testing random ideas, etc
devOut = outputs[n1:n2]
testIn = inputs[n2:] #  Test data (10%) 
testOut = outputs[n2:]

In [5]:
n_embd = 10 # the dimensionality of the character embedding vectors
n_hidden = 100 # the number of neurons in the hidden layer of the MLP
vocab_size = 27

# each FlattenConsecutive(2) concatenates 2 neighbours, so the next Linear sees 2x the features
model = Sequential([
  Embedding(vocab_size, n_embd),
  FlattenConsecutive(2), Linear(n_embd * 2, n_hidden, bias=False), BatchNorm1D(n_hidden), Tanh(),
  FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1D(n_hidden), Tanh(),
  FlattenConsecutive(2), Linear(n_hidden * 2, n_hidden, bias=False), BatchNorm1D(n_hidden), Tanh(),
  Linear(n_hidden, vocab_size),
])

# last layer: make less confident
model.layers[-1].weight.data *= 0.1
# all other layers: apply gain
for layer in model.layers[:-1]:
    if isinstance(layer, Linear):
        layer.weight.data *= 1.0 #5/3

parameters = model.parameters()
print(sum(p.data.size for p in parameters)) # number of parameters in total

45597
